In [3]:
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
pd.read_csv("C:\\Users\\ASUS\\Documents\\unified mentor\\p-2\\hhs_sheet.csv")
df=pd.DataFrame(pd.read_csv("C:\\Users\\ASUS\\Documents\\unified mentor\\p-2\\hhs_sheet.csv"))

In [5]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 6 columns):
 #   Column                                           Non-Null Count  Dtype
---  ------                                           --------------  -----
 0   Date                                             720 non-null    str  
 1   Children apprehended and placed in CBP custody*  720 non-null    int64
 2   Children in CBP custody                          720 non-null    int64
 3   Children transferred out of CBP custody          720 non-null    int64
 4   Children in HHS Care                             720 non-null    str  
 5   Children discharged from HHS Care                720 non-null    int64
dtypes: int64(4), str(2)
memory usage: 43.8 KB


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,12/21/2025,6,18,11,"2,484",14
1,12/18/2025,11,50,6,"2,472",16
2,12/17/2025,7,31,11,"2,481",10
3,12/16/2025,8,54,15,"2,468",9
4,12/15/2025,11,42,9,"2,470",7


In [6]:
df=df.dropna(subset=['Date']).copy()
df.shape

(720, 6)

In [7]:
df['Date']=pd.to_datetime(df['Date'],format='mixed',dayfirst=True)

In [8]:
df.head()

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,2025-12-21,6,18,11,"2,484",14
1,2025-12-18,11,50,6,"2,472",16
2,2025-12-17,7,31,11,"2,481",10
3,2025-12-16,8,54,15,"2,468",9
4,2025-12-15,11,42,9,"2,470",7


In [9]:
df=df.sort_values('Date').reset_index(drop=True)
df['report_gap']=(df['Date']-df['Date'].shift(1)).dt.days

In [10]:
df['report_gap'].value_counts().sort_index()

report_gap
1.0     503
2.0     105
3.0      88
4.0      16
5.0       4
6.0       2
11.0      1
Name: count, dtype: int64

In [11]:
df['lag1']=df['Children in HHS Care'].shift(1)
df['lag7']=df['Children in HHS Care'].shift(7)
df['lag14']=df['Children in HHS Care'].shift(14)

In [12]:
df[['Children in HHS Care','lag1','lag7','lag14']].head(15)
# df['Children in HHS Care'].dtype
df['Children in HHS Care']=df['Children in HHS Care'].str.replace(',','',regex=False).astype(float)


In [13]:
# rebuild lags fresh from the clean float column
df['lag1'] = df['Children in HHS Care'].shift(1)
df['lag7'] = df['Children in HHS Care'].shift(7)
df['lag14'] = df['Children in HHS Care'].shift(14)

# confirm all fixed
print(df['lag1'].dtype, df['lag7'].dtype, df['lag14'].dtype)

float64 float64 float64


In [14]:
df['rolling_mean_7']=df['Children in HHS Care'].shift(1).rolling(window=7).mean()
df['rolling_std_7']=df['Children in HHS Care'].shift(1).rolling(window=7).std()
df['rolling_mean_14']=df['Children in HHS Care'].shift(1).rolling(window=14).mean()
df['rolling_std_14']=df['Children in HHS Care'].shift(1).rolling(window=14).std()

In [15]:
df[['Children in HHS Care','rolling_mean_7','rolling_std_7','rolling_mean_14','rolling_std_14']].head(15)    

,Children in HHS Care,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14
0,7903.0,NaN,NaN,NaN,NaN
1,7930.0,NaN,NaN,NaN,NaN
2,8492.0,NaN,NaN,NaN,NaN
3,7002.0,NaN,NaN,NaN,NaN
4,8103.0,NaN,NaN,NaN,NaN
5,10598.0,NaN,NaN,NaN,NaN
6,9250.0,NaN,NaN,NaN,NaN
7,7122.0,8468.285714,1157.587249,NaN,NaN
8,7280.0,8356.714286,1254.714405,NaN,NaN
9,7433.0,8263.857143,1314.199430,NaN,NaN


In [16]:
df['net_stays']=df['Children transferred out of CBP custody']-df['Children discharged from HHS Care']
df['net_stays'].describe()

count    720.000000
mean     -44.738889
std       95.863856
min     -465.000000
25%      -94.250000
50%      -11.500000
75%        5.250000
max      206.000000
Name: net_stays, dtype: float64

In [17]:
df['day_of _the_week']=df['Date'].dt.dayofweek

df['day_of _the_week'].value_counts().sort_index()

day_of _the_week
0    130
1    132
2    128
3    135
4     36
5     35
6    124
Name: count, dtype: int64

In [18]:
df_model=df.dropna().reset_index(drop=True)
print(df_model.shape)

(706, 16)


In [19]:

df_model = df.dropna().reset_index(drop=True)
print("Cleaned shape:", df_model.shape)

test_size = 90

train = df_model.iloc[:-test_size]
test = df_model.iloc[-test_size:]

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train dates:", train['Date'].min(), "to", train['Date'].max())
print("Test dates:", test['Date'].min(), "to", test['Date'].max())

Cleaned shape: (706, 16)
Train shape: (616, 16)
Test shape: (90, 16)
Train dates: 2023-02-02 00:00:00 to 2025-08-09 00:00:00
Test dates: 2025-08-10 00:00:00 to 2025-12-21 00:00:00


In [20]:
from sklearn.metrics import mean_absolute_error
import numpy as np

n_splits = 5
test_window = 30  # each round tests on 30 rows

results = []

for i in range(n_splits):
    split_point = len(df_model) - test_window * (n_splits - i)
    
    train_fold = df_model.iloc[:split_point]
    test_fold = df_model.iloc[split_point : split_point + test_window]
    
    print(f"Round {i+1}: train up to {train_fold['Date'].max()}, test {test_fold['Date'].min()} to {test_fold['Date'].max()}")

Round 1: train up to 2025-05-05 00:00:00, test 2025-05-06 00:00:00 to 2025-06-26 00:00:00
Round 2: train up to 2025-06-26 00:00:00, test 2025-06-29 00:00:00 to 2025-08-09 00:00:00
Round 3: train up to 2025-08-09 00:00:00, test 2025-08-10 00:00:00 to 2025-09-21 00:00:00
Round 4: train up to 2025-09-21 00:00:00, test 2025-09-22 00:00:00 to 2025-11-02 00:00:00
Round 5: train up to 2025-11-02 00:00:00, test 2025-11-03 00:00:00 to 2025-12-21 00:00:00


In [21]:
from sklearn.metrics import mean_absolute_error
import numpy as np

n_splits = 5
test_window = 30
target = 'Children in HHS Care'

mae_scores = []

for i in range(n_splits):
    split_point = len(df_model) - test_window * (n_splits - i)
    
    train_fold = df_model.iloc[:split_point]
    test_fold = df_model.iloc[split_point : split_point + test_window]
    
    y_true = test_fold[target]
    y_pred = test_fold['lag1']
   
    mae = mean_absolute_error(y_true, y_pred)
    mae_scores.append(mae)
    print(f"Round {i+1}: MAE = {mae:.2f}")

print(f"\nAverage MAE across rounds: {np.mean(mae_scores):.2f}")

Round 1: MAE = 297.40
Round 2: MAE = 521.27
Round 3: MAE = 316.97
Round 4: MAE = 119.43
Round 5: MAE = 294.90

Average MAE across rounds: 309.99


In [22]:
mae_scores_ma = []

for i in range(n_splits):
    split_point = len(df_model) - test_window * (n_splits - i)
    
    train_fold = df_model.iloc[:split_point]
    test_fold = df_model.iloc[split_point : split_point + test_window]
    
    y_true = test_fold[target]
    y_pred = test_fold['rolling_mean_7']
    
    mae = mean_absolute_error(y_true, y_pred)
    mae_scores_ma.append(mae)
    print(f"Round {i+1}: MAE = {mae:.2f}")

print(f"\nAverage MAE (moving average): {np.mean(mae_scores_ma):.2f}")

Round 1: MAE = 427.61
Round 2: MAE = 448.35
Round 3: MAE = 326.75
Round 4: MAE = 95.23
Round 5: MAE = 259.55

Average MAE (moving average): 311.50


In [23]:
from statsmodels.tsa.arima.model import ARIMA

mae_scores_arima = []

for i in range(n_splits):
    split_point = len(df_model) - test_window * (n_splits - i)
    
    train_fold = df_model.iloc[:split_point]
    test_fold = df_model.iloc[split_point : split_point + test_window]
    
    model = ARIMA(train_fold[target], order=(1,1,1))
    fitted = model.fit()
    
    forecast = fitted.forecast(steps=len(test_fold))
    
    mae = mean_absolute_error(test_fold[target], forecast)
    mae_scores_arima.append(mae)
    print(f"Round {i+1}: MAE = {mae:.2f}")

print(f"\nAverage MAE (ARIMA): {np.mean(mae_scores_arima):.2f}")

Round 1: MAE = 279.18
Round 2: MAE = 498.14
Round 3: MAE = 379.12
Round 4: MAE = 102.12
Round 5: MAE = 217.41

Average MAE (ARIMA): 295.19


In [24]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

mae_scores_sarima = []

for i in range(n_splits):
    split_point = len(df_model) - test_window * (n_splits - i)
    
    train_fold = df_model.iloc[:split_point]
    test_fold = df_model.iloc[split_point : split_point + test_window]
    
    model = SARIMAX(train_fold[target], 
                     order=(1,1,1), 
                     seasonal_order=(1,1,1,5))
    fitted = model.fit(disp=False)
    
    forecast = fitted.forecast(steps=len(test_fold))
    
    mae = mean_absolute_error(test_fold[target], forecast)
    mae_scores_sarima.append(mae)
    print(f"Round {i+1}: MAE = {mae:.2f}")

print(f"\nAverage MAE (SARIMA): {np.mean(mae_scores_sarima):.2f}")

Round 1: MAE = 316.22
Round 2: MAE = 355.04


c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Round 3: MAE = 294.93


c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Round 4: MAE = 267.36
Round 5: MAE = 354.75

Average MAE (SARIMA): 317.66


In [25]:
from sklearn.ensemble import RandomForestRegressor

features = ['lag1', 'lag7', 'lag14', 
            'rolling_mean_7', 'rolling_std_7', 
            'rolling_mean_14', 'rolling_std_14',
            'net_stays', 'day_of _the_week', 'report_gap']

mae_scores_rf = []

for i in range(n_splits):
    split_point = len(df_model) - test_window * (n_splits - i)
    
    train_fold = df_model.iloc[:split_point]
    test_fold = df_model.iloc[split_point : split_point + test_window]
    
    X_train = train_fold[features]
    y_train = train_fold[target]
    X_test = test_fold[features]
    y_test = test_fold[target]
    
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores_rf.append(mae)
    print(f"Round {i+1}: MAE = {mae:.2f}")

print(f"\nAverage MAE (Random Forest): {np.mean(mae_scores_rf):.2f}")

Round 1: MAE = 211.99
Round 2: MAE = 289.50
Round 3: MAE = 291.37
Round 4: MAE = 112.39
Round 5: MAE = 150.83

Average MAE (Random Forest): 211.22


In [26]:
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# ---------- RMSE and MAPE using Random Forest ----------
rmse_scores = []
mape_scores = []

for i in range(n_splits):
    split_point = len(df_model) - test_window * (n_splits - i)
    train_fold = df_model.iloc[:split_point]
    test_fold = df_model.iloc[split_point : split_point + test_window]

    X_train = train_fold[features]
    y_train = train_fold[target]
    X_test = test_fold[features]
    y_test = test_fold[target]

    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = mean_absolute_percentage_error(y_test, y_pred)

    rmse_scores.append(rmse)
    mape_scores.append(mape)
    print(f"Round {i+1}: RMSE = {rmse:.2f}, MAPE = {mape:.2%}")

print(f"\nAverage RMSE: {np.mean(rmse_scores):.2f}")
print(f"Average MAPE: {np.mean(mape_scores):.2%}")

Round 1: RMSE = 373.30, MAPE = 7.27%
Round 2: RMSE = 577.72, MAPE = 10.69%
Round 3: RMSE = 399.04, MAPE = 13.17%
Round 4: RMSE = 238.95, MAPE = 4.59%
Round 5: RMSE = 218.95, MAPE = 6.29%

Average RMSE: 361.59
Average MAPE: 8.40%


In [27]:
# ---------- Horizon Error: accuracy at t+1 vs t+7 vs t+14 ----------
horizons = [1, 7, 14]
horizon_results = {}

for h in horizons:
    train_fold = df_model.iloc[:-h]
    test_fold = df_model.iloc[-h:]

    X_train = train_fold[features]
    y_train = train_fold[target]
    X_test = test_fold[features]
    y_test = test_fold[target]

    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae_h = mean_absolute_error(y_test, y_pred)
    horizon_results[h] = mae_h
    print(f"Horizon t+{h}: MAE = {mae_h:.2f}")

print("\nHorizon error summary:", horizon_results)

Horizon t+1: MAE = 72.16
Horizon t+7: MAE = 221.76
Horizon t+14: MAE = 211.56

Horizon error summary: {1: 72.1550000000002, 7: 221.75857142857149, 14: 211.55642857142854}


In [28]:
# ---------- Forecast Accuracy (%) ----------
# Simple version: 100% - MAPE
forecast_accuracy = 1 - np.mean(mape_scores)
print(f"Forecast Accuracy: {forecast_accuracy:.2%}")

Forecast Accuracy: 91.60%


In [29]:
capacity_threshold = df_model[target].quantile(0.90)
print("Capacity threshold:", capacity_threshold)

breach_probs = []

for i in range(n_splits):
    split_point = len(df_model) - test_window * (n_splits - i)
    
    train_fold = df_model.iloc[:split_point]
    test_fold = df_model.iloc[split_point : split_point + test_window]
    
    X_train = train_fold[features]
    y_train = train_fold[target]
    X_test = test_fold[features]
    
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)
    y_pred_fold = model.predict(X_test)
    
    breach = (y_pred_fold > capacity_threshold).mean()
    breach_probs.append(breach)
    print(f"Round {i+1}: Breach Probability = {breach:.2%}")

print(f"\nAverage Capacity Breach Probability: {np.mean(breach_probs):.2%}")

Capacity threshold: 9762.0
Round 1: Breach Probability = 0.00%
Round 2: Breach Probability = 0.00%
Round 3: Breach Probability = 0.00%
Round 4: Breach Probability = 0.00%
Round 5: Breach Probability = 0.00%

Average Capacity Breach Probability: 0.00%


In [30]:
stability_index = np.std(mae_scores_rf)
print(f"Forecast Stability Index (lower = more stable): {stability_index:.2f}")

Forecast Stability Index (lower = more stable): 72.06


In [31]:
df_model.columns.to_list()

['Date',
 'Children apprehended and placed in CBP custody*',
 'Children in CBP custody',
 'Children transferred out of CBP custody',
 'Children in HHS Care',
 'Children discharged from HHS Care',
 'report_gap',
 'lag1',
 'lag7',
 'lag14',
 'rolling_mean_7',
 'rolling_std_7',
 'rolling_mean_14',
 'rolling_std_14',
 'net_stays',
 'day_of _the_week']

In [34]:
df_model.to_excel("C:\\Users\\ASUS\\Documents\\unified mentor\\p-2\\hhs_features.xlsx", index=False)